In [1]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [2]:
## spatial join
# target_features = ?
# join_features = ?
# output_features = os.path.join(gdb, ?)

# fieldmappings = arcpy.FieldMappings()
# fieldmappings.addTable(target_features)
# fieldmappings.addTable(join_features)

# # variable
# fieldindex = fieldmappings.findFieldMapIndex(?)
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Sum'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
# sj_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [3]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [4]:
outputs = ['.\\Outputs']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])


In [5]:
p = pd.DataFrame.spatial.from_featureclass(r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels_pts')
p.columns

Index(['OBJECTID', 'Join_Count', 'TARGET_FID', 'parcel_id', 'WFRC_parcel_id',
       'county_id', 'CO_NAME', 'year_built', 'total_market_value',
       'land_value', 'building_id', 'building_type_id', 'building_type',
       'building_sqft', 'non_residential_sqft', 'residential_units',
       'job_spaces', 'stories', 'unit_price_non_residential',
       'res_price_per_sqft', 'basebldg', 'redev_friction', 'NoBuild', 'IS_OUG',
       'parcel_acres', 'Tax_Exempt', 'parent_parcel', 'volume_one_way',
       'volume_two_way', 'volume_two_way_nofwy', 'zonal_ppa', 'x', 'y', 'note',
       'parcel_sqft', 'Split', 'Split_Factor', 'MAG_parcel_id', 'max_far',
       'max_dua', 'type1', 'type2', 'type3', 'type4', 'type5', 'type6',
       'type7', 'type8', 'TAZID_900', 'distsml_id', 'distmed_id', 'distlrg_id',
       'CITY_NAME', 'stream_dist', 'streams', 'trail_dist', 'trail',
       'airport_distance', 'airport', 'fwy_exit_dist', 'fwy_exit_new',
       'bus_stop_dist_new', 'bus_stop_new', 'bus_rte

In [6]:
p_for_popsim = p[['parcel_id',  'CO_NAME', 'TAZID_900', 'year_built', 'residential_units', 'SHAPE']].copy()
mask = ((p_for_popsim['year_built'] <= 2023) | (p_for_popsim['year_built'].isna()) & p_for_popsim['residential_units'] > 0)
p_for_popsim = p_for_popsim[mask].copy()
p_for_popsim.columns = ['parcel_id',  'COUNTY', 'TAZID_900', 'APX_BLT_YR', 'UNIT_COUNT', 'SHAPE']
p_for_popsim['APX_BLT_YR'] = p_for_popsim['APX_BLT_YR'].fillna(0)

In [7]:
p_for_popsim.spatial.to_featureclass(location=os.path.join(outputs[0], 'parcels_20260130.shp'),sanitize_columns=False)

'e:\\Tasks\\REMM-Manage-Base-Year-Data-2023\\Inputs\\Tables\\Outputs\\parcels_20260130.shp'